In [0]:
import os
from pyspark.sql.functions import current_timestamp, input_file_name
import glob

# 1. Configuração (Caminho NATIVO do Volume, sem /dbfs)
volume_path = "/Volumes/olist_portfolio/bronze/raw_data"
catalog_schema = "olist_portfolio.bronze"

print(f"--- Buscando arquivos em: {volume_path} ---\n")

# 2. Listar arquivos usando o caminho direto
# No Serverless, o python consegue ler direto de /Volumes
files = glob.glob(f"{volume_path}/*.csv")

if not files:
    print("ERRO: Ainda não encontrei arquivos. Vamos tentar listar o diretório para ver o que tem lá:")
    try:
        print(os.listdir(volume_path))
    except Exception as e:
        print(f"Erro ao acessar diretório: {e}")
else:
    print(f"Encontrados {len(files)} arquivos CSV! Começando a carga...\n")

# 3. Loop de Ingestão
for file_path in files:
    try:
        # Pega o nome do arquivo (ex: /Volumes/.../olist_orders_dataset.csv -> olist_orders_dataset.csv)
        file_name = os.path.basename(file_path)
        
        # Limpa o nome para ficar bonito na tabela (ex: 'orders')
        table_name = file_name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
        
        print(f"Processando tabela: {table_name} ...")
        
        # Ler o CSV
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .option("multiLine", "true") \
            .option("quote", "\"") \
            .option("escape", "\"") \
            .load(file_path)
        
        # Adicionar metadados
        df_final = df.withColumn("ingestion_timestamp", current_timestamp())
        
        # Salvar tabela
        df_final.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"{catalog_schema}.{table_name}")
        
        print(f"  -> SUCESSO: Tabela {table_name} criada!")
        
    except Exception as e:
        print(f"  -> ERRO em {table_name}: {e}")

print("\n--- Fim do Processo ---")